In [40]:
from network_enum import NetworkEnum
import heapq
import itertools

class Receive:
    def __init__(self, ReceiverType=None, ReceiverID=None, TimeFullyReceived=None):
        self.ReceiverType = ReceiverType
        self.ReceiverID = ReceiverID
        self.TimeFullyReceived = TimeFullyReceived
        self.Latency = None

    def __repr__(self):
        return f"Receive(ReceiverType={self.ReceiverType}, ReceiverID={self.ReceiverID}, TimeFullyReceived={self.TimeFullyReceived}, Latency={self.Latency})"
    def calculate_latency(self, send_time):
        if self.TimeFullyReceived is not None and send_time is not None:
            self.Latency = self.TimeFullyReceived - send_time
        else:
            self.Latency = None
class Frame:
    def __init__(self, frame_nr, TimeSending=None, TimeFullySent=None, Receives=None):
        self.frame_nr = frame_nr
        self.TimeSending = TimeSending
        self.TimeFullySent = TimeFullySent
        # Receives is a list[Receive]
        self.Receives = Receives or []
        self.AverageLatency = None
    def __repr__(self):
        return f"Frame({self.frame_nr}, TimeSending={self.TimeSending}, TimeFullySent={self.TimeFullySent}, Receives={len(self.Receives)}, AverageLatency={self.AverageLatency})"
    def calculate_receive_latencies(self):
        self.AverageLatency = 0
        for receive in self.Receives:
            receive.calculate_latency(self.TimeSending)
            self.AverageLatency += receive.Latency if receive.Latency is not None else 0
        self.AverageLatency = self.AverageLatency / len(self.Receives) if self.Receives else None
class Track:
    def __init__(self, track_id):
        self.track_id = track_id
        self.Frames = []  # list[Frame]

    def __repr__(self):
        return f"Track({self.track_id}, Frames={len(self.Frames)})"

class LogParser:
    def __init__(self, file_path):
        self.file_path = file_path

    def parse_logs(self):
        """Yield parsed log entries from the file."""
        with open(self.file_path, 'r') as file:
            for line in file:
                yield self._parse_line(line.strip())

    def filter_logs(self, filter_id=None, entries=None):
        """Yield entries from `entries` (an iterable of parsed entries) or from
        `self.parse_logs()` if entries is None. If filter_id is provided, only
        yield entries whose 'id' field equals filter_id.
        """
        if entries is None:
            entries = self.parse_logs()
        for entry in entries:
            if filter_id is None or entry.get('id') == filter_id:
                yield entry

    @staticmethod
    def merge_files(file_paths):
        """Merge parsed log entries from multiple files by ascending timestamp ('ts').

        - file_paths: iterable of file path strings (relative to current cwd or absolute).
        - Entries without a numeric 'ts' are ordered after those with timestamps.
        - Returns a generator that yields parsed entries in ascending ts order.
        """
        # Create an iterator for each file's parsed entries
        iterators = []
        for fp in file_paths:
            parser = LogParser(fp)
            iterators.append(parser.parse_logs())

        # Initialize heap with first item from each iterator
        heap = []
        counter = itertools.count()

        for it_idx, it in enumerate(iterators):
            try:
                entry = next(it)
            except StopIteration:
                continue
            ts = entry.get('ts')
            try:
                ts_val = int(ts) if ts is not None else float('inf')
            except Exception:
                ts_val = float('inf')
            heapq.heappush(heap, (ts_val, next(counter), entry, it))

        while heap:
            ts_val, _, entry, it = heapq.heappop(heap)
            yield entry
            try:
                nxt = next(it)
            except StopIteration:
                continue
            ts = nxt.get('ts')
            try:
                nxt_ts = int(ts) if ts is not None else float('inf')
            except Exception:
                nxt_ts = float('inf')
            heapq.heappush(heap, (nxt_ts, next(counter), nxt, it))

    def build_tracks(self, entries=None):
        """Build Track objects grouped by trackID.

        - Only entries that contain a 'trackID' field are considered.
        - For each frame (grouped by trackID + frame number) we collect
          TimeSending and TimeFullySent using NetworkEnum.FrameSending and
          NetworkEnum.FrameFullySent (timestamps come from the 'ts' field).
        - For entries with NetworkEnum.FrameFullyRecv we collect per-receiver
          records (receiverType + receiverID -> TimeFullyRecv). Multiple
          receives per frame are supported.

        Returns a list of Track objects with Frame objects in ascending frame order.
        """
        if entries is None:
            # only consider SFUConnection entries by default
            entries = self.filter_logs(filter_id='SFUConnection')

        tracks_map = {}  # trackID -> frame_nr -> {'TimeSending':..., 'TimeFullySent':..., 'Receives': {(rtype,rid): ts}}

        for entry in entries:
            if entry["id"] != "SFUConnection":
                continue
            if 'trackID' not in entry:
                continue
            track_id = entry.get('trackID')
            frame_val = entry.get('frame')
            if frame_val is None:
                continue
            try:
                frame_nr = int(frame_val)
            except Exception:
                # keep as-is if not int
                frame_nr = frame_val

            tracks_map.setdefault(track_id, {})
            frame_bucket = tracks_map[track_id].setdefault(frame_nr, {'TimeSending': None, 'TimeFullySent': None, 'Receives': {}})

            status = entry.get('status')
            ts = entry.get('ts')
            # try to coerce timestamp to int if present
            try:
                ts = int(ts) if ts is not None else None
            except Exception:
                pass

            if status == NetworkEnum.FrameSending:
                frame_bucket['TimeSending'] = ts
            elif status == NetworkEnum.FrameFullySent:
                frame_bucket['TimeFullySent'] = ts
            elif status == NetworkEnum.FrameFullyRecv:
                # collect receiver info; logs are expected to have receiverType and receiverID
                rtype = entry.get('receiverType')
                rid = entry.get('receiverID')
                # Fallback names that may appear in logs
                if rtype is None:
                    rtype = entry.get('receiver')
                if rid is None:
                    rid = entry.get('receiverId') if 'receiverId' in entry else entry.get('receiverID')
                key = (rtype, rid)
                # store the latest timestamp for this receiver+frame (or the first)
                frame_bucket['Receives'][key] = ts

        # Convert to Track and Frame objects
        tracks = []
        for track_id, frames_dict in tracks_map.items():
            track = Track(track_id)
            for fr_nr in sorted(frames_dict.keys()):
                data = frames_dict[fr_nr]
                # convert Receives dict into list of Receive objects
                receives_list = []
                for (rtype, rid), ts_val in data.get('Receives', {}).items():
                    receives_list.append(Receive(ReceiverType=rtype, ReceiverID=rid, TimeFullyReceived=ts_val))
                frame_obj = Frame(fr_nr, TimeSending=data.get('TimeSending'), TimeFullySent=data.get('TimeFullySent'), Receives=receives_list)
                track.Frames.append(frame_obj)
            tracks.append(track)

        return tracks

    def _parse_line(self, line):
        fields = {}
        for field in line.split():
            key, value = field.split('=', 1)
            if key == "status":
                # convert numeric status into NetworkEnum if possible
                try:
                    fields[key] = NetworkEnum(int(value))
                except Exception:
                    fields[key] = value
            else:
                fields[key] = value
        return fields

# Example usage:
# parser = LogParser('path_to_log_file.log')
# for log_entry in parser.parse_logs():
#     print(log_entry['id'])  # Access the 'id' field
#     print(log_entry)        # Access all fields
#sfup_log_20250904_171552


In [ ]:
parser = LogParser('sfup_cl1_log_20251014_172435')
for log_entry in parser.parse_logs():
    print(log_entry['id'])  # Access the 'id' field
    print(log_entry)        # Access all fields

In [58]:
# Demonstrate merging two log files and building tracks from the merged stream
merged_entries = LogParser.merge_files(['sfup_cl0_log_20251015_163032', 'sfup_cl1_log_20251015_163032'])
tracks = LogParser('sfup_cl1_log_20251015_163032').build_tracks(entries=merged_entries)
for track in tracks:
    if track.track_id == "cl0_mdc_video_0_0":
        continue
    print(track)
    for f in track.Frames:
        f.calculate_receive_latencies()
        print('  ', f)
        print(f.Receives)
    if track.Frames:
        print('  Receives for first frame:', track.Frames[0].Receives)


Track(cl1_mdc_video_0_0, Frames=23)
   Frame(0, TimeSending=1760538633644, TimeFullySent=1760538633648, Receives=1, AverageLatency=16.0)
[Receive(ReceiverType=None, ReceiverID=None, TimeFullyReceived=1760538633660, Latency=16)]
   Frame(100, TimeSending=1760538636945, TimeFullySent=1760538636949, Receives=1, AverageLatency=6.0)
[Receive(ReceiverType=None, ReceiverID=None, TimeFullyReceived=1760538636951, Latency=6)]
   Frame(200, TimeSending=1760538640245, TimeFullySent=1760538640248, Receives=1, AverageLatency=16.0)
[Receive(ReceiverType=None, ReceiverID=None, TimeFullyReceived=1760538640261, Latency=16)]
   Frame(300, TimeSending=1760538643545, TimeFullySent=1760538643549, Receives=1, AverageLatency=7.0)
[Receive(ReceiverType=None, ReceiverID=None, TimeFullyReceived=1760538643552, Latency=7)]
   Frame(400, TimeSending=1760538646845, TimeFullySent=1760538646848, Receives=1, AverageLatency=7.0)
[Receive(ReceiverType=None, ReceiverID=None, TimeFullyReceived=1760538646852, Latency=7)]
  